# Autonomous Transform Example

Use the extension sidebar to select any dataset file, then click **Apply** with this transformation selected.
This cell is designed to run with server-injected variables (`df`, `dataset_path`) so it always follows the dataset currently selected in the UI.

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

# The extension injects df and dataset_path at apply-time.
if "df" not in locals():
    raise RuntimeError(
        "`df` is not defined. Run this through the extension Apply button so the selected dataset is injected."
    )

selected_dataset = Path(str(locals().get("dataset_path", "unknown")))
working = df.copy()

# Normalize column names to make downstream use more consistent.
working.columns = [str(col).strip().replace(" ", "_") for col in working.columns]
working = working.dropna(how="all")

# Add provenance fields to verify which selected dataset was used.
working["__source_dataset"] = selected_dataset.as_posix()
working["__source_file"] = selected_dataset.name
working["__applied_at_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

sort_column = "RespondentID" if "RespondentID" in working.columns else working.columns[0]
working = working.sort_values(sort_column, kind="stable")

result = working.reset_index(drop=True)

display(result.head(10))
print(f"Applied transform to selected dataset: {selected_dataset}")
print(f"Rows before: {len(df)}, rows after: {len(result)}")
